# Local PPT Key-Data Demo

This notebook is designed for local demonstration alongside the presentation deck. It checks the raw data, recomputes the EDA numbers shown in the PPT, and displays the local validation evidence that supports the report. The Kaggle `0.81716` public submission remains in `notebooks/demo_081716_xgb.ipynb`.

## 1. Setup

In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path = Path.cwd()) -> Path:
    for path in [start, *start.parents]:
        if (path / "data" / "train.csv").exists() and (path / "code" / "local_validation_reproduction.py").exists():
            return path
    raise FileNotFoundError("Open this notebook from inside the repository folder.")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
TABLE_DIR = PROJECT_ROOT / "experiments" / "tables"
sys.path.insert(0, str(PROJECT_ROOT / "code"))

print("Project root:", PROJECT_ROOT)
print("Data files:", sorted(path.name for path in DATA_DIR.glob("*.csv")))

## 2. Dataset Numbers From The PPT

In [ ]:
import pandas as pd
import numpy as np

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

group_id = train["PassengerId"].str[:4]
group_size = group_id.value_counts()
group_labels = train.assign(GroupId=group_id).groupby("GroupId")["Transported"]

dataset_ppt_check = pd.DataFrame([
    ["Training rows", len(train), "8,693"],
    ["Test rows", len(test), "4,277"],
    ["Raw predictive columns", train.drop(columns=["Transported"]).shape[1], "13"],
    ["Positive class rate", f"{train['Transported'].mean() * 100:.2f}%", "50.36%"],
    ["Largest missing rate", f"{train.isna().mean().max() * 100:.2f}%", "2.50%"],
    ["Multi-passenger row rate", f"{group_id.map(group_size).gt(1).mean() * 100:.1f}%", "44.7%"],
    ["Group-homogeneous rate", f"{group_labels.nunique().eq(1).mean() * 100:.2f}%", "87.18%"],
], columns=["Metric", "Local recomputation", "PPT value"])

display(dataset_ppt_check)

## 3. EDA Signals From The PPT

In [ ]:
cabin = train["Cabin"].str.split("/", expand=True)
train_eda = train.copy()
train_eda["Deck"] = cabin[0]
spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
train_eda["TotalSpend"] = train_eda[spend_cols].fillna(0).sum(axis=1)
train_eda["AgeBand"] = pd.cut(train_eda["Age"], bins=[-1, 12, 18, 25, 40, 60, 200])

deck_rates = train_eda.groupby("Deck")["Transported"].mean().sort_values(ascending=False)
planet_rates = train_eda.groupby("HomePlanet")["Transported"].mean().sort_values(ascending=False)
spend_zero_rates = train_eda.assign(NoSpend=train_eda["TotalSpend"].eq(0)).groupby("Transported")["NoSpend"].mean()

eda_ppt_check = pd.DataFrame([
    ["Deck B transport rate", f"{deck_rates.loc['B'] * 100:.0f}%", "73%"],
    ["Deck T transport rate", f"{deck_rates.loc['T'] * 100:.0f}%", "20%"],
    ["Europa transport rate", f"{planet_rates.loc['Europa'] * 100:.0f}%", "66%"],
    ["Earth transport rate", f"{planet_rates.loc['Earth'] * 100:.0f}%", "42%"],
    ["No-spend rate if transported", f"{spend_zero_rates.loc[True] * 100:.1f}%", "PPT: spend signal"],
    ["No-spend rate if not transported", f"{spend_zero_rates.loc[False] * 100:.1f}%", "PPT: spend signal"],
], columns=["Signal", "Local recomputation", "PPT value"])

display(eda_ppt_check)
display(deck_rates.rename("transport_rate").reset_index())
display(planet_rates.rename("transport_rate").reset_index())

## 4. Feature Engineering And Validation Protocol

In [ ]:
from local_validation_reproduction import (
    AUDIT_SEEDS,
    MODEL_SEARCH_FEATURES,
    N_SPLITS,
    SEARCH_SEEDS,
    SIMPLE_FEATURES,
    add_model_search_features,
)

train_fe = add_model_search_features(train)

protocol = pd.DataFrame([
    ["Splitter", "Repeated StratifiedGroupKFold", "Slide 7"],
    ["Group key", "PassengerId prefix", "Slide 7"],
    ["Folds", N_SPLITS, "Slide 7"],
    ["Search seeds", ", ".join(map(str, SEARCH_SEEDS)), "7, 21, 42, 87, 123"],
    ["Audit seeds", ", ".join(map(str, AUDIT_SEEDS)), "3, 11, 17, 29, 57, 68, 99, 131"],
    ["Simple feature count", len(SIMPLE_FEATURES), "22-feature local line"],
    ["Model-search feature count", len(MODEL_SEARCH_FEATURES), "31-feature local line"],
], columns=["Item", "Notebook value", "PPT / report reference"])

display(protocol)
display(train_fe[MODEL_SEARCH_FEATURES].head())

## 5. Key Results Aligned With PPT

In [ ]:
benchmark = pd.read_csv(TABLE_DIR / "model_benchmark.csv")
ablation = pd.read_csv(TABLE_DIR / "feature_ablation.csv")
cv_public = pd.read_csv(TABLE_DIR / "cv_vs_public.csv")
tuning = pd.read_csv(TABLE_DIR / "tuning_summary.csv")
xgb_reconstruction = pd.read_csv(TABLE_DIR / "xgb_branch_reconstruction.csv")

print("Slide 8: five-model benchmark")
display(benchmark)

print("Slide 11: feature-family ablation")
display(ablation)

print("Slide 12: tuning gains")
display(tuning)

print("Slide 13: local CV versus public leaderboard")
display(cv_public)

print("Slide 15: XGBoost branch reconstruction")
display(xgb_reconstruction)

## 6. Optional Full Local Validation Retraining

The notebook above is the fast local demo aligned with the PPT. If the teacher asks to see the local validation experiments retrained, uncomment the cell below. It reruns the consolidated LightGBM/XGBoost validation code and can take several minutes on a laptop.

In [ ]:
# from local_validation_reproduction import run_reproduction
# reproduced = run_reproduction()
# display(reproduced)